# Cached Prediction EDA

Streamlined analysis notebook for cached LORO outputs with all-genes model fits in harmonized evaluation space.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eval_utils.results_eda import *


In [ ]:
import importlib
import src.eval_utils.results_eda
importlib.reload(src.eval_utils.results_eda)

In [ ]:
set_academic_style()

CFG = EDAConfig(
    csv_path='data/raw/gxp_samples.csv',
    hvg_path='data/raw/ahba_100hvg.txt',
    cache_root='out/loro_subject_cache',  # or 'out/loro_subject_cache_arxiv'
    gene_scope='allgenes',
    plam_cache_dirname='plam',
)
CFG
print('analysis scope:', CFG.gene_scope)

## 1) Dataset Attributes


In [ ]:
PREPOST = prepare_pre_post_harmonization(CFG)
print('subjects:', len(PREPOST['subjects']))
print('genes:', len(PREPOST['genes']))
print('parcels:', PREPOST['raw_cube'].shape[1])


In [ ]:
# label_mode: 'ahba' | 'gtex' | 'both'
LABEL_MODE = 'both'
ELIGIBLE_ONLY = True

count_df = parcel_subject_count_table(PREPOST, label_mode=LABEL_MODE, eligible_only=ELIGIBLE_ONLY)
n_pool = int(count_df['n_subjects_pool'].iloc[0])
print(f'n (subjects in pool) = {n_pool}; eligible_only={ELIGIBLE_ONLY}')
count_df[['parcel_idx','ahba_parcel','gtex_native','n_subjects_observed','n_samples','label']].head(12)


In [ ]:
fig_h = max(8.0, 0.20 * len(count_df))
fig, ax = plot_parcel_subject_counts(count_df, top_n=None, min_subjects=1, figsize=(10, 10))


In [ ]:
# LORO-relevant: x=regions sampled per subject, y=subject count
subj_cov = subject_region_count_distribution(PREPOST, eligible_only=ELIGIBLE_ONLY)
display(subj_cov)
fig, ax = plot_subject_region_count_distribution(subj_cov)


In [ ]:
CSV_PATH = "data/raw/gxp_samples.csv"
gxp_samples_df = pd.read_csv(CSV_PATH)


In [ ]:
# from data.utils import (
#     build_and_write_hvg_gene_list,
#     build_and_write_tissue_deg_gene_list,
# )

# # GTEx HVGs: default tissue-discriminative
# out_path, ranked = build_and_write_hvg_gene_list(
#     samples_df=gxp_samples_df,
#     dataset="GTEx",
#     method="variance",
#     n_genes=100,
# )
# print("wrote", out_path)

# # GTEx HVGs: within-tissue / demeaned alternative
# out_path, ranked = build_and_write_hvg_gene_list(
#     samples_df=gxp_samples_df,
#     dataset="GTEx",
#     method="demeaned",
#     n_genes=100,
#     out_name=f"gtex_100hvg_demeaned.txt",
# )
# print("wrote", out_path)

# # GTEx tissue-enriched DEG panel: top 10 per tissue
# deg_path, deg_ranked = build_and_write_tissue_deg_gene_list(
#     samples_df=gxp_samples_df,
#     dataset="GTEx",
#     top_k_per_group=25,
#     out_name="gtex_25deg.txt",
# )
# print("wrote", deg_path)
# display(deg_ranked.head(24))


## 2) Pre/Post ComBat Diagnostics


In [ ]:
# gene_mode: 'hvg' | 'allgenes' | 'custom'
GENE_MODE = 'hvg'
CUSTOM_GENE_LIST = None

atlas_cmp = prepare_atlas_median_comparison(
    PREPOST,
    gene_mode=GENE_MODE,
    gene_list=CUSTOM_GENE_LIST,
    observed_only=True,
)
print('atlas parcels:', len(atlas_cmp['parcel_labels']))
print('genes in panel:', len(atlas_cmp['genes']))


In [ ]:
fig, axes = plot_atlas_median_comparison_heatmaps(atlas_cmp, label_stride=1)


In [ ]:
# Single-subject pre/post (GTEx only)
SUBJECT_ID = PREPOST['subjects'][0]
fig, axes = plot_subject_prepost_heatmaps(
    PREPOST,
    subject_id=SUBJECT_ID,
    gene_mode=GENE_MODE,
    gene_list=CUSTOM_GENE_LIST,
    observed_only=True,
    label_stride=1,
)
print('subject:', SUBJECT_ID)


In [ ]:
fig, axes = plot_region_region_spearman_heatmaps(atlas_cmp)


In [ ]:
# Optional population-level region-wise subject-subject pre/post view
avail = available_regions(PREPOST, min_subjects=1)
REGION = int(avail.iloc[0]['parcel_idx'])
fig, axes = plot_region_covariance_side_by_side(PREPOST, region=REGION, mode='subject')


## 3) Performance across Gene Panels


In [ ]:
import importlib
import src.eval_utils.results_eda as results_eda
importlib.reload(results_eda)
from src.eval_utils.results_eda import *

from scipy.stats import ttest_rel
from itertools import combinations

MODELS = ['naive', 'dlam', 'plam']
N_JOBS = 16


def paired_ttests_by_subject(metrics_df, label=''):
    pivot = metrics_df.pivot(index='subject', columns='model', values='pearson_r').dropna()
    model_names = pivot.columns.tolist()
    t_results = {}
    print(f"\nPaired t-tests for {label}:")
    for model1, model2 in combinations(model_names, 2):
        t_stat, p_val = ttest_rel(pivot[model1], pivot[model2])
        t_results[(model1, model2)] = {'t_stat': t_stat, 'p_val': p_val}
        print(f"  {model1} vs {model2}: t = {t_stat:.4f}, p = {p_val:.4g}")
    return pivot, t_results


def run_subject_metric_panel(label, eval_gene_path=None):
    dfs = [
        compute_subject_metrics_from_cache_gene_subset(
            CFG,
            model=model,
            eval_gene_path=eval_gene_path,
            n_jobs=N_JOBS,
        )
        for model in MODELS
    ]
    metrics_df = pd.concat(dfs, ignore_index=True)
    display(metrics_df.head())
    fig, axes, summary_df = plot_loro_subject_summary_bars(metrics_df, use_sem=True)
    display(summary_df)
    pivot_df, t_results = paired_ttests_by_subject(metrics_df, label=label)
    display(pivot_df.head())
    return metrics_df, summary_df, pivot_df, t_results

### All gene performance



In [ ]:
metrics_allgenes_eval, summary_allgenes_eval, pivot_allgenes, t_results_allgenes = run_subject_metric_panel(
    label='all genes',
)

### AHBA 100 HVG



In [ ]:
metrics_ahba100hvg_eval, summary_ahba100hvg_eval, pivot_ahba100hvg, t_results_ahba100hvg = run_subject_metric_panel(
    label='ahba_100hvg',
    eval_gene_path='ahba_100hvg',
)

### GTEx 100 HVG



In [ ]:
metrics_gtex100hvg_eval, summary_gtex100hvg_eval, pivot_gtex100hvg, t_results_gtex100hvg = run_subject_metric_panel(
    label='gtex_100hvg',
    eval_gene_path='gtex_100hvg',
)

### GTEx 120 DEG



In [ ]:
metrics_gtex25deg_eval, summary_gtex25deg_eval, pivot_gtex25deg, t_results_gtex25deg = run_subject_metric_panel(
    label='gtex_25deg',
    eval_gene_path='gtex_25deg',
)

### Richiardi 2015 gene list
Correlated gene expression supports synchronous activity in brain networks (Science, 2015)



In [ ]:
metrics_richiardi_eval, summary_richiardi_eval, pivot_richiardi, t_results_richiardi = run_subject_metric_panel(
    label='richiardi2015',
    eval_gene_path='richiardi2015',
)

### Synaptic gene ontology list



In [ ]:
metrics_syngo_eval, summary_syngo_eval, pivot_syngo, t_results_syngo = run_subject_metric_panel(
    label='syngo',
    eval_gene_path='syngo',
)

## 4) Fold-Combo Difficulty



In [ ]:
fold_perf_df, combo_df = compute_fold_combo_metrics_from_cache(
    CFG,
    PREPOST,
    coverage_min=5,
    coverage_max=12,
    models=MODELS,
    n_jobs=N_JOBS,
)
print('fold_perf_df:', fold_perf_df.shape)
print('combo_df:', combo_df.shape)
display(combo_df.head())

In [ ]:
for model_name in MODELS:
    fig, ax, key_table = plot_fold_combo_ranked(
        combo_df,
        PREPOST,
        model=model_name,
        metric='mean_pearson',
        style='points',
        points_style='dist_to_nearest_train',
        show_running_average=False,
        running_average_window=20,
    )
    display(key_table)

In [ ]:
heldout_df = summarize_heldout_region_performance(fold_perf_df, PREPOST)
fig, ax, heldout_plot_df = plot_heldout_region_grouped_bars(
    heldout_df,
    metric='mean_pearson',
    sort_by_model='plam',
    use_error_bars=True,
    error_kind='sem',
)
display(heldout_df.head(20))